In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))
from configs.config import FAISS_INDEX_DIR, NOTEBOOKS_DIR

# --- Papermill parameters (overwritten at runtime) ---
faiss_index_path = str(FAISS_INDEX_DIR)
embedding_model  = "all-MiniLM-L6-v2"
llm_model        = "Qwen/Qwen1.5-0.5B"
retriever_k      = 5
temperature      = 0.7
max_new_tokens   = 512
top_k            = 50
experiment_name  = "default_retrieve_run"
run_id           = "default_run"

In [ ]:
# Incluir esto al comienzo del notebook (después de la celda de parámetros si usas papermill)
import mlflow

# Asegurar que estamos en el run correcto sin iniciar uno nuevo
if mlflow.active_run() is None and "run_id" in globals():
    mlflow.start_run(run_id=run_id)


In [ ]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

from src.utils import build_embedder, extract_answer

mlflow.log_params({
    "faiss_index_path": faiss_index_path,
    "embedding_model":  embedding_model,
    "llm_model":        llm_model,
    "retriever_k":      retriever_k,
    "temperature":      temperature,
    "max_new_tokens":   max_new_tokens,
    "top_k":            top_k,
})

custom_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a helpful assistant specialised in financial reports.
Use ONLY the following context to answer the question as accurately and concisely as possible.
If the answer is not present in the context, respond with: "I don't know".

Context:
{context}

Question:
{question}

Answer:""",
)

# build_embedder handles BGE query-prefix automatically
embedder = build_embedder(embedding_model)

db = FAISS.load_local(
    folder_path=faiss_index_path,
    embeddings=embedder,
    allow_dangerous_deserialization=True,
)
retriever = db.as_retriever(search_kwargs={"k": retriever_k})
mlflow.log_metric("index_docs", len(db.index_to_docstore_id))

tokenizer = AutoTokenizer.from_pretrained(llm_model)
model     = AutoModelForCausalLM.from_pretrained(llm_model, device_map="auto")
pipe      = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=max_new_tokens,
    do_sample=True,
    top_k=top_k,
    temperature=temperature,
)
llm = HuggingFacePipeline(pipeline=pipe)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": custom_prompt},
)

'\nYou are a helpful assistant specialized in financial reports. \nUse ONLY the following context to answer the question as accurately and concisely as possible.\n\nIf the answer is not present in the context, respond with: "I don’t know".\n\nContext:\nAt December 31, 2018, 3M had $3.3 billion of cash, cash equivalents and marketable securities, of which approximately $3.1 billion was held by the Company’s foreign subsidiaries and approximately $160 million was held by the United States. These balances are invested in bank instruments and other high-quality fixed income securities. At December\xa031, 2017, cash, cash equivalents and marketable securities held by the Company’s foreign subsidiaries and by the United States totaled approximately $3.975 billion\n\n3M expects to contribute approximately $100 million to $200 million of cash to its global defined benefit pension and postretirement plans in 2019. The Company does not have a required minimum cash pension contribution obligation

In [ ]:
mlflow.end_run()